# Reranker Ölçümü (Colab) — hybrid vs +reranker

PC'de `rerank_hazirla.py` her sorgu için hybrid top-50 adayı çıkardı → `rerank_input.jsonl`.
Burada reranker (T4 GPU) o 50 adayı yeniden sıralar → hybrid vs +reranker metriği.

**ÖNCE:** Runtime → Change runtime type → **T4 GPU**. Sonra `rerank_input.jsonl`'i Drive'a yükle.

4 GB laptop'ta reranker+embed sığmıyordu; T4 (16 GB) rahat + hızlı.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip install -q transformers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

GIRDI = '/content/drive/MyDrive/mevzuat/rerank_input.jsonl'   # KENDI yoluna gore duzelt
import os, json
print('var mi:', os.path.exists(GIRDI))

veri = []
with open(GIRDI, encoding='utf-8') as f:
    for line in f:
        veri.append(json.loads(line))
print(f'{len(veri)} sorgu yuklendi')
print('ilk sorgu aday sayisi:', len(veri[0]['adaylar']))

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL = 'BAAI/bge-reranker-v2-m3'
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL).cuda().eval()
print('reranker yuklendi (T4 GPU)')

In [ ]:
# Her sorgu icin: reranker adaylari puanla + yeniden sirala
# HEM hybrid (mevcut sira) HEM reranked sirayi hesapla
import time
t0 = time.time()

def rerank_skorla(sorgu, metinler, batch=32):
    skorlar = []
    for i in range(0, len(metinler), batch):
        parca = metinler[i:i+batch]
        ciftler = [[sorgu, m] for m in parca]
        with torch.no_grad():
            inp = tok(ciftler, padding=True, truncation=True, max_length=512, return_tensors='pt').to('cuda')
            s = model(**inp).logits.view(-1).float().cpu().tolist()
        skorlar.extend(s)
    return skorlar

hybrid_ranks, rerank_ranks = [], []
for j, v in enumerate(veri):
    dogru = tuple(v['dogru'])
    adaylar = [(a['kn'], a['mn']) for a in v['adaylar']]
    # hybrid sira (girdi zaten hybrid siralı)
    h = next((i for i, a in enumerate(adaylar, 1) if a == dogru), 0)
    hybrid_ranks.append(h)
    # reranker
    metinler = [a['text'] for a in v['adaylar']]
    skor = rerank_skorla(v['sorgu'], metinler)
    sirali = [a for a, _ in sorted(zip(adaylar, skor), key=lambda x: -x[1])]
    r = next((i for i, a in enumerate(sirali, 1) if a == dogru), 0)
    rerank_ranks.append(r)
    if (j+1) % 100 == 0:
        print(f'  {j+1}/{len(veri)}  ({time.time()-t0:.0f}s)', end='\r')

print(f'\nBitti ({time.time()-t0:.0f}s)')

In [ ]:
import math
def R(ranks, k): return sum(1 for r in ranks if 1 <= r <= k) / len(ranks)
def MRR(ranks): return sum((1.0/r) for r in ranks if r > 0) / len(ranks)
def nDCG(ranks): return sum((1.0/math.log2(r+1)) for r in ranks if 1 <= r <= 10) / len(ranks)

n = len(hybrid_ranks)
print('='*58)
print(f'RERANKER OLCUMU — {n} sorgu (aday: top-50)')
print('='*58)
print(f"{'Metrik':<12}{'Hybrid':>12}{'+Reranker':>12}{'Fark':>10}")
print('-'*58)
for k in [1, 5, 10]:
    h, r = R(hybrid_ranks, k), R(rerank_ranks, k)
    print(f'Recall@{k:<6}{h:>12.4f}{r:>12.4f}{r-h:>+10.4f}')
print(f"{'MRR':<12}{MRR(hybrid_ranks):>12.4f}{MRR(rerank_ranks):>12.4f}{MRR(rerank_ranks)-MRR(hybrid_ranks):>+10.4f}")
print(f"{'nDCG@10':<12}{nDCG(hybrid_ranks):>12.4f}{nDCG(rerank_ranks):>12.4f}{nDCG(rerank_ranks)-nDCG(hybrid_ranks):>+10.4f}")
print('='*58)
print('Fark POZITIF+buyukse -> reranker ise yariyor, kalici yap.')
print('Fark kucuk/negatifse -> reranker gereksiz, kaldir.')